Starting place for, do some imports and define some variables.

Update the following Variables

*   PROJECT_ID
*   LOCATION
*   STAGING_BUCKET




In [ ]:
# 1. Install required packages with the correct extras
!pip install --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk requests googlemaps

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://chal5_stage_bucket"

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c


Define Agent for a basic Google Search

In [ ]:
#Cell 2
import vertexai
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, google_search_tool
from vertexai.preview import reasoning_engines
from typing import Optional


google_search_agent = Agent(
    name="Google_Search_Bot",
    description="General information search agent.",
    instruction="Use Google Search to find information about any topic.",
    tools=[google_search_tool.GoogleSearchTool()]
)


app = reasoning_engines.AdkApp(agent=google_search_agent)
print('System Initiated')

System Initiated


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


This is a test for the ADKApp local instance to ensure the agent is working as intended.

In [ ]:
# Test the local ADKApp instance using stream_query with the required user_id
try:
    print("Starting local stream query...")
    responses = app.stream_query(
        user_id="local-test-user",
        message="What are the current top headlines in world sports?",
    )
    for response in responses:
        print(response)
except Exception as e:
    print(f"Local test failed with error: {e}")

Starting local stream query...
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Here are some of the current top headlines in world sports:\n\n**Football (Soccer)**\n\nThe FIFA World Cup 2026 is currently a major focus, with the quarterfinals approaching and analysis of the round of 16 taking center stage. Morocco\'s World Cup quarterfinal hopes have been dealt a blow with an injury to Saibari. There\'s been controversy surrounding refereeing decisions, with Egypt reportedly criticizing "influential refereeing" in their match against Argentina. Lionel Messi is leading the Golden Boot race ahead of Kylian Mbappe and Erling Haaland. Off the pitch, Justin Bieber is slated to co-headline the World Cup final halftime show, and FIFA has condemned a racist attack by an Argentina fan on IShowSpeed. France also lost an appeal regarding Olise\'s yellow card. Looking ahead, Mexico has appointed Rafael Márquez as their new national team coach after their World Cup exit. There 

This will deploy the Agent to the Agent Platform
*Pinning of versions was needed to overcome compatibility issues between Agent Platform runtime environment.*

In [ ]:
import vertexai
from vertexai import agent_engines

# Pinning the versions to match your local environment for consistency
remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.157.0",
        "google-adk==1.29.0"
    ],
)

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.157.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==1.157.0', 'google-adk==1.29.0', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket chal5_stage_bucket
INFO:vertexai.agent_engines:Wrote to gs://chal5_stage_bucket/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/239267613896/locations/us

Run a test againt the remote Agent Platform deployment to ensure functionality.

In [ ]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="Give me the news highlights in the world of sports.",
):
  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Here\'s a rundown of recent highlights in the world of sports:\n\n**Football (Soccer):**\n\nThe FIFA World Cup 2026 is generating significant buzz with several key matches and controversies. Argentina secured a thrilling victory against Egypt in the Round of 16, with Lionel Messi scoring a superb equalizer. However, this match was not without controversy, as an Egypt goal was disallowed after VAR spotted a foul against Argentina\'s Lisandro Martínez. There\'s also been a strong reaction to officiating in this match, with the Egypt boss criticizing it as "entirely undeserved". Kylian Mbappé continues to be a central figure, though he has faced insults from a Paraguayan senator. Rafael Márquez has been named the Mexican National Team Head Coach for the 2030 World Cup. Off the pitch, Justin Bieber is slated to co-headline the 2026 World Cup Final Halftime Show. In transfer news, Newcastle captain Bruno Guimarães has rep